In [ ]:
!pip install pyspark

# **loading the data set**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pyspark.sql import SparkSession


In [ ]:
spark = SparkSession.builder.appName("SparkByExamples.com").getOrCreate()

In [ ]:
arti_clean = spark.read.parquet("/content/drive/MyDrive/h&m_project/articles_cleaned.parquet")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cust_clean = spark.read.parquet("/content/drive/MyDrive/h&m_project/customers_cleaned.parquet")
#

In [ ]:
tran_interaction = spark.read.parquet("/content/drive/MyDrive/h&m_project/tran_interaction.parquet")

In [ ]:
tran_filter = spark.read.parquet("/content/drive/MyDrive/h&m_project/transactions_cleaned.parquet")

In [ ]:
user_profile = spark.read.parquet("/content/drive/MyDrive/h&m_project/user_profiles.parquet")

In [ ]:
print('DATA LOAD SUCCESFULLY')

In [ ]:
tran_interaction.columns

In [ ]:
user_profile.columns

In [ ]:
cust_clean.columns

# **JOINING THE TABLES**

In [ ]:
master_df = tran_interaction.join(user_profile, on="customer_id", how="inner")

In [ ]:
master_df.columns

In [ ]:
master_df = master_df.join(cust_clean, on="customer_id", how="inner")

In [ ]:
master_df = master_df.join(arti_clean, on="article_id", how="inner")

In [ ]:
master_df.columns

In [ ]:
master_df.show()

# Re-Indexing the article_id and customer_id

> Add blockquote



In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
cust_window = Window.orderBy("customer_id")
master_df = master_df.withColumn("user_idx", F.row_number().over(cust_window) - 1)

In [ ]:
arti_window = Window.orderBy("article_id")
master_df = master_ai.withColumn("item_idx", F.row_number().over(arti_window) - 1)

**# selecting the final columns**

In [ ]:
final_cols = [
      "user_idx",
      "item_idx",
      "purchase_count",
      "age_group",
      "avg_spending",
      "popularity_score"]

In [ ]:
final_df = master_df.select(final_cols)

## **# saving the final_dataframe**

In [ ]:
final_df.write.mode("overwrite").parquet("/content/drive/MyDrive/h&m_project/ai_input.parquet")
print("Master Dataset saved! Ready for AI Modeling.")